# RAG sur le Code des personnes et de la famille du Togo

## Objectif
Construire un système de question-réponse (RAG) permettant d'interroger le Code togolais de la famille en langage naturel.

## 1. Installation des dépendances (à exécuter une fois)

```python
!pip install sentence-transformers chromadb langchain pypdf ollama

In [6]:
!pip install sentence-transformers chromadb langchain langchain-text-splitters ollama

   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   ---------------------------------------- 588.9/588.9 kB 4.3 MB/s  0:00:00
   ---------------------------------------- 0.0/10.8 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/10.8 MB 6.3 MB/s eta 0:00:02
   --------- ------------------------------ 2.6/10.8 MB 6.3 MB/s eta 0:00:02
   -------------- ------------------------- 3.9/10.8 MB 6.3 MB/s eta 0:00:02
   ------------------- -------------------- 5.2/10.8 MB 6.3 MB/s eta 0:00:01
   ------------------------ --------------- 6.6/10.8 MB 6.3 MB/s eta 0:00:01
   ---------------------------- ----------- 7.6/10.8 MB 6.3 MB/s eta 0:00:01
   --------------------------------- ------ 8.9/10.8 MB 6.3 MB/s eta 0:00:01
   ------------------------------------- -- 10.2/10.8 MB 6.3 MB/s eta 0:00:01
   ---------------------------------------- 10.8/10.8 MB 6.2 MB/s  0:00:01
   ---------------------------------------- 0.0/671.5 kB ? eta -:--:--
   ----------------

In [1]:
# 1. Importations
import os
import re
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
import ollama
import warnings
warnings.filterwarnings('ignore')

print("✅ Toutes les bibliothèques sont chargées")


✅ Toutes les bibliothèques sont chargées


In [2]:
# 2. Chargement du document
with open("code_des_personnes_et_de_la_famille.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"📄 Taille du document : {len(raw_text)} caractères")


📄 Taille du document : 277782 caractères


In [3]:
# 3. Fonction de découpage manuel (sans langchain)
def split_text_into_chunks(text, chunk_size=1200, overlap=200):
    """Découpe le texte en chunks avec chevauchement"""
    parts = re.split(r'(?=\nArticle \d+|\nTITRE [IVXLCDM]+|\nCHAPITRE)', text)
    
    chunks = []
    current_chunk = ""
    
    for part in parts:
        if len(part) > chunk_size:
            paragraphs = part.split('\n\n')
            for para in paragraphs:
                if len(current_chunk) + len(para) <= chunk_size:
                    current_chunk += para + "\n\n"
                else:
                    if current_chunk:
                        chunks.append(current_chunk.strip())
                    current_chunk = para + "\n\n"
        else:
            if len(current_chunk) + len(part) <= chunk_size:
                current_chunk += part
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = part
        
        if len(current_chunk) > chunk_size:
            chunks.append(current_chunk[:chunk_size])
            current_chunk = current_chunk[-overlap:] if overlap > 0 else ""
    
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    chunks = [c for c in chunks if len(c) > 50]
    return chunks


In [4]:
# 4. Découpage
chunks = split_text_into_chunks(raw_text, chunk_size=1200, overlap=200)
print(f"📦 Nombre de chunks : {len(chunks)}")

📦 Nombre de chunks : 286


In [5]:
# 5. Création de la base vectorielle ChromaDB
client = chromadb.PersistentClient(path="./chroma_db_code")

try:
    client.delete_collection("code_famille")
    print("🗑️ Ancienne collection supprimée")
except:
    pass

collection = client.create_collection(
    name="code_famille",
    embedding_function=embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name='all-MiniLM-L6-v2'
    )
)
print("✅ Collection ChromaDB créée")

🗑️ Ancienne collection supprimée


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Collection ChromaDB créée


In [6]:
# 6. Indexation des chunks
batch_size = 100
for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i+batch_size]
    batch_ids = [f"chunk_{j}" for j in range(i, i+len(batch_chunks))]
    collection.add(
        documents=batch_chunks,
        ids=batch_ids
    )
    print(f"✅ Ajouté {len(batch_chunks)} chunks (total: {min(i+len(batch_chunks), len(chunks))}/{len(chunks)})")

print(f"\n✅ Indexation terminée. {collection.count()} chunks dans la base.")


✅ Ajouté 100 chunks (total: 100/286)
✅ Ajouté 100 chunks (total: 200/286)
✅ Ajouté 86 chunks (total: 286/286)

✅ Indexation terminée. 286 chunks dans la base.


In [7]:
# 7. Fonction de recherche
def search(query, top_k=3):
    results = collection.query(query_texts=[query], n_results=top_k)
    return results['documents'][0]


In [8]:
# 8. Test de recherche
test_query = "Quels sont les cas de nullité du mariage ?"
passages = search(test_query)
print(f"\n🔍 Question : {test_query}")
for i, p in enumerate(passages):
    print(f"   Passage {i+1} : {p[:150]}...")


🔍 Question : Quels sont les cas de nullité du mariage ?
   Passage 1 : Article 83 : La nullité du mariage doit être prononcée : 
1°- lorsqu’il a été contracté sans le consentement de l’un des 
époux; 
2°- lorsque les conj...
   Passage 2 : Article 84 : L’action en nullité fondée sur les dispositions de l’article 
précédent peut être exercée : 
- par le ministère public ; 
- par les époux...
   Passage 3 : Article 54 : Le mariage est prohibé entre parents: 
1) en ligne directe, à tous les degrés ; 
2) en ligne collatérale, entre frère et sœur, oncle et n...


In [9]:
# 9. Vérification d'Ollama
def check_ollama():
    try:
        # Méthode alternative pour vérifier Ollama
        import subprocess
        result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
        if result.returncode == 0:
            print("✅ Ollama est accessible")
            # Extraire les noms des modèles
            lines = result.stdout.strip().split('\n')[1:]  # Ignorer l'en-tête
            models = [line.split()[0] for line in lines if line.strip()]
            print(f"📦 Modèles disponibles : {models}")
            return True
        else:
            print("⚠️ Ollama n'est pas accessible")
            return False
    except Exception as e:
        print(f"⚠️ Erreur : {e}")
        return False

In [10]:
import ollama

# Test simple
try:
    response = ollama.generate(model='llama3.2:3b', prompt='Dis "Bonjour" en une phrase courte')
    print("✅ Génération réussie !")
    print(f"Réponse : {response['response']}")
except Exception as e:
    print(f"❌ Erreur : {e}")

✅ Génération réussie !
Réponse : "Bonjour" est un salut traditionnel et poli utilisé pour s'adresse à quelqu'un, généralement en français.


In [22]:
def check_ollama():
    try:
        import subprocess
        result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
        if result.returncode == 0:
            print("✅ Ollama est accessible")
            lines = result.stdout.strip().split('\n')[1:]
            models = [line.split()[0] for line in lines if line.strip()]
            print(f"📦 Modèles disponibles : {models}")
            return True
    except Exception as e:
        print(f"⚠️ Erreur : {e}")
    return False

check_ollama()

⚠️ Erreur : [WinError 2] Le fichier spécifié est introuvable


False

In [11]:
import ollama
import requests

# Vérification 1 : Appel direct via Python
try:
    response = ollama.generate(model='llama3.2:3b', prompt='Bonjour')
    print("✅ Ollama fonctionne !")
    print(f"Réponse : {response['response'][:100]}...")
except Exception as e:
    print(f"❌ Erreur : {e}")

# Vérification 2 : Appel API direct
try:
    resp = requests.get("http://localhost:11434/api/tags")
    if resp.status_code == 200:
        models = resp.json().get('models', [])
        print(f"📦 Modèles disponibles : {[m['name'] for m in models]}")
except Exception as e:
    print(f"⚠️ API directe : {e}")

✅ Ollama fonctionne !
Réponse : Bonjour ! Comment puis-je vous aider aujourd'hui ?...
📦 Modèles disponibles : ['llama3.2:3b']


In [12]:
# 10. Fonction de génération (si Ollama est disponible)
def generate_with_context(query, context_chunks, model="llama3.2:3b"):
    context = "\n\n---\n\n".join(context_chunks)
    if len(context) > 3000:
        context = context[:3000] + "..."
    
    prompt = f"""Tu es un expert en droit de la famille togolais. 
Réponds à la question en te basant UNIQUEMENT sur le contexte fourni.
Si la réponse n'est pas dans le contexte, dis "Je n'ai pas trouvé cette information dans le code."

### CONTEXTE ###
{context}

### QUESTION ###
{query}

### RÉPONSE ###
"""
    
    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
            options={'temperature': 0.3}
        )
        return response['message']['content']
    except Exception as e:
        return f"Erreur : {e}"


In [13]:
# 11. Pipeline RAG
def rag(query, top_k=3):
    passages = search(query, top_k=top_k)
    answer = generate_with_context(query, passages)
    return answer, passages


In [14]:
# 12. Test final
if check_ollama():
    question = "Quel est l'âge minimum pour se marier ?"
    print(f"\n📌 Question : {question}")
    reponse, sources = rag(question)
    print(f"💡 Réponse : {reponse}")

✅ Ollama est accessible
📦 Modèles disponibles : ['llama3.2:3b']

📌 Question : Quel est l'âge minimum pour se marier ?
💡 Réponse : Je n'ai pas trouvé cette information dans le code. Le code ne mentionne pas explicitement un âge minimum pour se marier. Cependant, il y a des articles qui traitent de la paternité et de l'âge des enfants, mais pas d'âge minimum pour le mariage.


In [15]:
# Vérifier ce que la recherche trouve
query = "âge minimum pour se marier"
results = collection.query(query_texts=[query], n_results=3)

print("🔍 Résultats de la recherche :")
for i, doc in enumerate(results['documents'][0]):
    print(f"\n--- Passage {i+1} ---")
    print(doc[:500])

🔍 Résultats de la recherche :

--- Passage 1 ---
Article 153 : Les enfants seront confiés à la femme jusqu’à l’âge de sept 
ans à moins que le tribunal, sur la demande du mari, ou à défaut, du 
conseil de famille ou du ministère public et au vu des conclusions d’une 
enquête sociale, n’ordonne dans l’intérêt des enfants, que tous ou 
quelques-uns d’entre eux seront confiés aux soins soit du mari, soit 
d’une tierce personne. 
Lorsque les enfants seront âgés de plus de sept (07) ans, le tribunal 
ordonnera, en fonction de leur intérêt, que tous

--- Passage 2 ---
Article 43 : La loi reconnaît la monogamie et la polygamie. 
L’option est déclarée par les époux dans les conditions fixées par l’article 
51. 
Toutefois, la monogamie est la forme de mariage de droit commun.

Section 1ère : Conditions de fond 

Article 44 : L’homme et la femme choisissent librement leur conjoint et 
ne contractent mariage que de leur libre et plein consentement. 
L’homme et la femme avant dix huit (18) ans ne 

In [16]:
question = "À partir de quel âge un homme et une femme peuvent-ils contracter mariage ?"
reponse, sources = rag(question)

print(f"📌 Question : {question}")
print(f"💡 Réponse : {reponse}\n")

📌 Question : À partir de quel âge un homme et une femme peuvent-ils contracter mariage ?
💡 Réponse : Selon l'article 44, un homme et une femme ne peuvent contracter mariage que si chacun d'eux a au moins 18 ans. Cependant, le président du tribunal ou le juge aux affaires matrimoniales peut accorder des dispenses d'âge pour des motifs sérieux, mais cette dispense ne peut pas être accordée si l'un des époux a moins de 16 ans.

